# UK Income Tax Reform Simulation
## 1. Introduction
This notebook simulates the fiscal and distributional effects of a proposed UK personal income tax reform.
It removes the current tax-free personal allowance and replaces the progressive structure with a generic two-tier or three-tier flat tax system without deductions or reliefs.

**Key Questions Addressed**:
1. What is the baseline simulated income tax take under the current system?
2. What is the simulated tax take under each reform scenario?
3. What is the gross change in tax revenue before behavioural adjustments?
4. What is the net change in tax revenue after behavioural adjustments?
5. Which deciles gain or lose, and by how much?
6. How does the reform affect low, middle, and high-income households?
7. How sensitive are results to behavioural assumptions?
8. How do outcomes differ between the two-tier and three-tier designs?

## 2. Parameters
Define the baseline tax system, reform parameters, and behavioural assumptions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional for interactive charts, disabled by default to keep simple
# import plotly.express as px

# Set display options
pd.options.display.float_format = '{:,.2f}'.format

# --- Baseline System Parameters ---
BASELINE_PA = 12570  # Personal Allowance
BASELINE_BASIC_THRESHOLD = 50270
BASELINE_HIGHER_THRESHOLD = 125140

BASELINE_BASIC_RATE = 0.20
BASELINE_HIGHER_RATE = 0.40
BASELINE_ADDITIONAL_RATE = 0.45

# --- Reform Scenarios Parameter Sets ---
# Two-tier flat tax scenario
REFORM_TWO_TIER = {
    'thresholds': [50000],          # First tier up to £50,000
    'rates': [0.20, 0.35]           # 20% below threshold, 35% above
}

# Three-tier flat tax scenario
REFORM_THREE_TIER = {
    'thresholds': [20000, 80000],   # Tiers at £20k and £80k
    'rates': [0.15, 0.25, 0.35]     # 15%, 25%, 35%
}

# --- Behavioural Settings ---
# Elasticity of Taxable Income (ETI)
ETI_CENTRAL = 0.2
ETI_LOW = 0.1
ETI_HIGH = 0.4

# Choose current modeling ETI
CURRENT_ETI = ETI_CENTRAL

## 3. Data Input and Validation
Load synthetic taxpayer data. If microdata is unavailable, this block generates a synthetic log-normal distribution representative of income.

In [ ]:
# Generate synthetic log-normal distribution for 100,000 taxpayers
np.random.seed(42)
POPULATION_SIZE = 100000

# Mean and sigma for lognormal roughly simulating UK income distribution
mean_log_income = np.log(30000)
sigma_log_income = 0.7

synthetic_incomes = np.random.lognormal(mean=mean_log_income, sigma=sigma_log_income, size=POPULATION_SIZE)

df = pd.DataFrame({
    'taxpayer_id': range(1, POPULATION_SIZE + 1),
    'gross_income_annual': synthetic_incomes,
    'weight': 1 # Assuming equal weight for synthetic data
})

# Basic validation
df = df[df['gross_income_annual'] >= 0].copy()
print(f"Total taxpayers: {len(df):,}")
print(f"Median Income: £{df['gross_income_annual'].median():,.2f}")
print(f"Mean Income: £{df['gross_income_annual'].mean():,.2f}")

## 4. Baseline Tax Engine
Computing tax liabilities under the current baseline progressive rules. Personal allowance taper for >£100k is modelled conventionally.

In [ ]:
def calculate_baseline_tax(income):
    # Standard personal allowance taper: £1 reduction for every £2 above £100k
    taper = max(0, (income - 100000) / 2) if income > 100000 else 0
    personal_allowance = max(0, BASELINE_PA - taper)
    
    taxable_income = max(0, income - personal_allowance)
    
    # Calculate bands
    basic_band = max(0, min(taxable_income, BASELINE_BASIC_THRESHOLD - personal_allowance))
    higher_band = max(0, min(taxable_income - basic_band, BASELINE_HIGHER_THRESHOLD - BASELINE_BASIC_THRESHOLD))
    additional_band = max(0, taxable_income - basic_band - higher_band)
    
    tax = (basic_band * BASELINE_BASIC_RATE) + \
          (higher_band * BASELINE_HIGHER_RATE) + \
          (additional_band * BASELINE_ADDITIONAL_RATE)
          
    return tax

df['baseline_tax'] = df['gross_income_annual'].apply(calculate_baseline_tax)
df['baseline_post_tax'] = df['gross_income_annual'] - df['baseline_tax']
df['baseline_mtr'] = np.where(df['gross_income_annual'] > BASELINE_HIGHER_THRESHOLD, BASELINE_ADDITIONAL_RATE,
                              np.where(df['gross_income_annual'] > BASELINE_BASIC_THRESHOLD, BASELINE_HIGHER_RATE,
                                       np.where(df['gross_income_annual'] > BASELINE_PA, BASELINE_BASIC_RATE, 0.0)))
                             
total_baseline_tax = (df['baseline_tax'] * df['weight']).sum()
print(f"Total Baseline Tax Take: £{total_baseline_tax / 1e9:,.2f} billion")

## 5. Reform Tax Engine
Functions to compute tax liabilities under multi-tier flat tax systems with no personal allowance or deductions.

In [ ]:
def calculate_reform_tax(income, thresholds, rates):
    tax = 0.0
    previous_threshold = 0
    
    for i, rate in enumerate(rates):
        if i < len(thresholds):
            threshold = thresholds[i]
            if income > previous_threshold:
                taxable_in_band = min(income - previous_threshold, threshold - previous_threshold)
                tax += taxable_in_band * rate
            previous_threshold = threshold
        else: # Top tier
            if income > previous_threshold:
                tax += (income - previous_threshold) * rate
                
    return tax

def get_reform_mtr(income, thresholds, rates):
    for i, threshold in enumerate(thresholds):
        if income <= threshold:
            return rates[i]
    return rates[-1]

# Apply Two-Tier Scenario A
df['reform_two_tier_tax'] = df['gross_income_annual'].apply(
    lambda x: calculate_reform_tax(x, REFORM_TWO_TIER['thresholds'], REFORM_TWO_TIER['rates']))
df['reform_two_tier_mtr'] = df['gross_income_annual'].apply(
    lambda x: get_reform_mtr(x, REFORM_TWO_TIER['thresholds'], REFORM_TWO_TIER['rates']))

# Apply Three-Tier Scenario B
df['reform_three_tier_tax'] = df['gross_income_annual'].apply(
    lambda x: calculate_reform_tax(x, REFORM_THREE_TIER['thresholds'], REFORM_THREE_TIER['rates']))
df['reform_three_tier_mtr'] = df['gross_income_annual'].apply(
    lambda x: get_reform_mtr(x, REFORM_THREE_TIER['thresholds'], REFORM_THREE_TIER['rates']))

## 6. Behavioural Response Engine
Using Elasticity of Taxable Income (ETI) to adjust reported gross income based on changes in the marginal net-of-tax rate.

In [ ]:
def apply_behavioural_response(df, baseline_mtr_col, reform_mtr_col, eti):
    # Net of tax rates
    net_of_tax_baseline = 1 - df[baseline_mtr_col]
    net_of_tax_reform = 1 - df[reform_mtr_col]
    
    # Fractional change in net-of-tax rate
    pct_change_net_of_tax = (net_of_tax_reform - net_of_tax_baseline) / np.clip(net_of_tax_baseline, 0.01, 1.0)
    
    # Behavioural adjustment factor
    adjustment_factor = 1 + (eti * pct_change_net_of_tax)
    
    # Prevent extreme values
    adjustment_factor = np.clip(adjustment_factor, 0.1, 2.0) 
    
    return df['gross_income_annual'] * adjustment_factor

# Calculate behavioural response for Two-Tier
df['gross_income_annual_two_tier_resp'] = apply_behavioural_response(
    df, 'baseline_mtr', 'reform_two_tier_mtr', CURRENT_ETI)

df['reform_two_tier_tax_resp'] = df['gross_income_annual_two_tier_resp'].apply(
    lambda x: calculate_reform_tax(x, REFORM_TWO_TIER['thresholds'], REFORM_TWO_TIER['rates']))

# Calculate behavioural response for Three-Tier
df['gross_income_annual_three_tier_resp'] = apply_behavioural_response(
    df, 'baseline_mtr', 'reform_three_tier_mtr', CURRENT_ETI)

df['reform_three_tier_tax_resp'] = df['gross_income_annual_three_tier_resp'].apply(
    lambda x: calculate_reform_tax(x, REFORM_THREE_TIER['thresholds'], REFORM_THREE_TIER['rates']))

## 7. Distributional Analysis
Assigning taxpayers to income deciles based on baseline gross income. Aggregating results.

In [ ]:
# Quantile based labels
df['decile'] = pd.qcut(df['gross_income_annual'], 10, labels=range(1, 11))

def aggregate_by_decile(df, tax_col, tax_resp_col):
    agg = df.groupby('decile', observed=False).agg(
        avg_gross_income=('gross_income_annual', 'mean'),
        baseline_tax=('baseline_tax', 'mean'),
        reform_tax=(tax_col, 'mean'),
        reform_tax_resp=(tax_resp_col, 'mean'),
    )
    agg['change_in_tax'] = agg['reform_tax'] - agg['baseline_tax']
    agg['percent_change_tax'] = (agg['change_in_tax'] / np.clip(agg['baseline_tax'], 1, None)) * 100
    agg['baseline_post_tax'] = agg['avg_gross_income'] - agg['baseline_tax']
    agg['reform_post_tax'] = agg['avg_gross_income'] - agg['reform_tax']
    agg['change_post_tax'] = agg['reform_post_tax'] - agg['baseline_post_tax']
    
    agg['baseline_atr'] = agg['baseline_tax'] / np.clip(agg['avg_gross_income'], 1, None)
    agg['reform_atr'] = agg['reform_tax'] / np.clip(agg['avg_gross_income'], 1, None)
    
    return agg

decile_summary_two_tier = aggregate_by_decile(df, 'reform_two_tier_tax', 'reform_two_tier_tax_resp')
decile_summary_three_tier = aggregate_by_decile(df, 'reform_three_tier_tax', 'reform_three_tier_tax_resp')

print("--- Two-Tier Decile Summary ---")
display(decile_summary_two_tier[['avg_gross_income', 'baseline_tax', 'reform_tax', 'change_in_tax']].head(10))

## 8. Summary Outputs & Visualisations
Aggregate metrics summarising overall fiscal and distributional impacts.

In [ ]:
# --- Headline Summary Metrics ---
def generate_summary(df, scenario_name, tax_col, tax_resp_col):
    baseline_revenue = df['baseline_tax'].sum()
    static_revenue = df[tax_col].sum()
    resp_revenue = df[tax_resp_col].sum()
    
    return pd.Series({
        'Scenario': scenario_name,
        'Baseline Revenue (£bn)': baseline_revenue / 1e9,
        'Reform Revenue (Static) (£bn)': static_revenue / 1e9,
        'Reform Revenue (Behavioural) (£bn)': resp_revenue / 1e9,
        'Gross Revenue Change (£bn)': (static_revenue - baseline_revenue) / 1e9,
        'Behavioural Offset (£bn)': (resp_revenue - static_revenue) / 1e9,
        'Net Revenue Change (£bn)': (resp_revenue - baseline_revenue) / 1e9
    })

summary_df = pd.DataFrame([
    generate_summary(df, 'Two-Tier Flat Tax', 'reform_two_tier_tax', 'reform_two_tier_tax_resp'),
    generate_summary(df, 'Three-Tier Flat Tax', 'reform_three_tier_tax', 'reform_three_tier_tax_resp')
])

print("--- Overall Revenue Summary ---")
display(summary_df)

# --- Visualisations ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Tax Change by Decile (Two-Tier)
axes[0].bar(decile_summary_two_tier.index, decile_summary_two_tier['change_in_tax'], color='skyblue')
axes[0].set_title('Average Tax Change by Decile (Two-Tier)')
axes[0].set_xlabel('Income Decile')
axes[0].set_ylabel('Change in Tax Paid (£)')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xticks(range(1, 11))

# Plot Average Tax Rate Change
axes[1].plot(decile_summary_two_tier.index, decile_summary_two_tier['baseline_atr']*100, label='Baseline ATR', marker='o')
axes[1].plot(decile_summary_two_tier.index, decile_summary_two_tier['reform_atr']*100, label='Reform ATR', marker='x')
axes[1].set_title('Average Tax Rate (ATR) Before and After (Two-Tier)')
axes[1].set_xlabel('Income Decile')
axes[1].set_ylabel('Effective Tax Rate (%)')
axes[1].legend()
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.show()

## 9. Sensitivity Analysis
Running alternative behavioural elasticity assumptions.

In [ ]:
# Compute under HIGH elasticity
df['gross_income_two_resp_high'] = apply_behavioural_response(
    df, 'baseline_mtr', 'reform_two_tier_mtr', ETI_HIGH)
df['reform_two_tax_resp_high'] = df['gross_income_two_resp_high'].apply(
    lambda x: calculate_reform_tax(x, REFORM_TWO_TIER['thresholds'], REFORM_TWO_TIER['rates']))

# Compute under LOW elasticity
df['gross_income_two_resp_low'] = apply_behavioural_response(
    df, 'baseline_mtr', 'reform_two_tier_mtr', ETI_LOW)
df['reform_two_tax_resp_low'] = df['gross_income_two_resp_low'].apply(
    lambda x: calculate_reform_tax(x, REFORM_TWO_TIER['thresholds'], REFORM_TWO_TIER['rates']))

sensitivity_df = pd.DataFrame([
    {'Assumption': 'ETI LOW (0.1)', 'Net Revenue Change (£bn)': (df['reform_two_tax_resp_low'].sum() - total_baseline_tax)/1e9},
    {'Assumption': 'ETI CENTRAL (0.2)', 'Net Revenue Change (£bn)': (df['reform_two_tier_tax_resp'].sum() - total_baseline_tax)/1e9},
    {'Assumption': 'ETI HIGH (0.4)', 'Net Revenue Change (£bn)': (df['reform_two_tax_resp_high'].sum() - total_baseline_tax)/1e9}
])

print("--- Sensitivity Iterations (Two-Tier) ---")
display(sensitivity_df)

## 10. Conclusions / Interpretation
This illustrative notebook establishes baseline metrics to gauge UK income tax reform feasibility. 

*   **Simplifying Deductions & Allowances:** Complete removal of the personal allowance massively expands the tax base but drastically alters distribution, disproportionately affecting lowest-income quintiles based on the replacement structural tiering.
*   **Decile Impacts:** Deciles where baseline Effective Tax Rate (ATR) drops will largely represent "winners", while those with an ATR hike face immediate tax-burden increases.
*   **Fiscal Revenue vs. Behaviour:** Higher top marginal rates can precipitate significant behavioral revenue erosion, underscoring the necessity of sound ETI modeling.

*Caveats: This model does not implement full complexity like interacting benefits, NI contributions, married couples allowances, and uses purely synthetic/indicative tax bases unless populated with real microdata.*